In [2]:
#Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re


In [3]:
#Read the file and pick the specific sheet that I want 
excelSheet = pd.ExcelFile('/Users/preciousajilore/Documents/GitHub/torchmtlr/data/finalforAIpredictionmodelfeb18.xlsx')
print(excelSheet.sheet_names)

df = pd.read_excel(
                  excelSheet,         #  name of the excel file
                  sheet_name='Sheet1', #   name of the sheet you want to read
                  engine='openpyxl')

['Sheet1']


In [4]:
"""
This functiom returns a DataFrame summarising what df.info() prints.
"""

def info_as_table(df):
    """
    Return a DataFrame summarising what df.info() prints.
    Columns:
      col            – column name
      non-null       – # non-missing entries
      null           – # missing entries
      % null         – percentage missing
      dtype          – pandas dtype
      mem_kB         – memory used by the column (≈, index excluded)
    """
    non_null = df.count()
    nulls    = df.isna().sum()
    mem_kB   = df.memory_usage(index=False) / 1024  # KiB
    out = (
        pd.concat([non_null, nulls, mem_kB, df.dtypes], axis=1)
          .set_axis(["non_null", "null", "mem_kB", "dtype"], axis=1)
          .assign(**{"% null": lambda x: (x["null"] / len(df) * 100).round(1)})
          .reset_index(names="col")
    )
    return out[["col", "non_null", "null", "% null", "dtype", "mem_kB"]]

In [5]:
summary = info_as_table(df)
summary.head()
summary.style.format({"% null": "{:.1f}%", "mem_kB": "{:.1f}"})

,col,non_null,null,% null,dtype,mem_kB
0,UAH ID,59,2007,97.1%,object,16.1
1,DOB (yyyy-mm-dd),2066,0,0.0%,object,16.1
2,Date of Surgery,2066,0,0.0%,datetime64[ns],16.1
3,monthsurg,1973,93,4.5%,float64,16.1
4,Age (at time of surgery),2065,1,0.0%,float64,16.1
5,Age: <50 = 0; >= 50= 1,1973,93,4.5%,object,16.1
6,stxlocation,2066,0,0.0%,object,16.1
7,distal,2065,1,0.0%,float64,16.1
8,penile,2064,2,0.1%,float64,16.1
9,StxEtiology,2065,1,0.0%,object,16.1


In [6]:
#   Inspect current dtypes
print(df[["DOB (yyyy-mm-dd)", "Date of Surgery"]].dtypes)
# → you'll see object / datetime64[ns]

#  Convert DOB to datetime (coerce bad rows to NaT)
df["DOB (yyyy-mm-dd)"] = pd.to_datetime(
        df["DOB (yyyy-mm-dd)"],        # source column
        format="%Y-%m-%d",             # strict ≈ faster; drop if mixed formats
        errors="coerce"                # anything un-parseable → NaT
)

# Quick sanity peek
print(df["DOB (yyyy-mm-dd)"].head())

DOB (yyyy-mm-dd)            object
Date of Surgery     datetime64[ns]
dtype: object
0   1953-04-04
1   1956-05-30
2   1949-05-19
3   1945-11-18
4   1928-11-07
Name: DOB (yyyy-mm-dd), dtype: datetime64[ns]


In [7]:
#This is to fix the ages of patients at the time of surgery
age_calc = ((df["Date of Surgery"] - df["DOB (yyyy-mm-dd)"])
            .dt.days // 365.25)              # whole-year age
df["Age_calc"] = age_calc.astype("Int16")

In [8]:
#Flag mismatches
age_diff   = df["Age (at time of surgery)"] - df["Age_calc"] #calculate the difference between the two columns
bad_mask   = age_diff.abs() > 1 #check if the difference is greater than 1 year
n_bad      = bad_mask.sum() #number of bad rows
pct_bad    = n_bad / len(df) * 100 #percentage of bad rows

print(f"{n_bad} rows differ by > 1 year "
      f"({pct_bad:.1f}% of all records).")

223 rows differ by > 1 year (10.8% of all records).


In [9]:
#   Inspect a few problem lines
cols = ["DOB (yyyy-mm-dd)", "Date of Surgery",
        "Age (at time of surgery)", "Age_calc"]
display(df.loc[bad_mask, cols].head(10))

,DOB (yyyy-mm-dd),Date of Surgery,Age (at time of surgery),Age_calc
18,1973-09-26,2004-02-03,36.0,30
76,1964-11-06,2005-03-14,45.0,40
77,1942-03-25,2005-03-14,58.0,62
79,1963-02-18,2005-03-28,46.0,42
80,1944-06-22,2005-03-28,65.0,60
109,1942-03-25,2005-11-05,59.0,63
164,1955-10-22,2006-08-11,53.0,50
194,1917-08-15,2007-01-06,85.0,89
198,1930-08-02,2007-02-06,72.0,76
199,1936-10-29,2007-02-06,66.0,70


CLEANING CYSTO COLUMN

NOTES: Change the empty cells and the ones that say "n/a","none" to 0

In [ ]:
df.columns 

Index(['UAH ID', 'DOB (yyyy-mm-dd)', 'Date of Surgery', 'monthsurg',
       'Age (at time of surgery)', 'Age: <50 = 0; >= 50= 1', 'stxlocation',
       'distal', 'penile', 'StxEtiology', 'StxLength', '# Strictures',
       'charlsons', 'Cormorbidity', 'Diabetes ', 'COPD ', 'Smoker ', 'BMI35+',
       'BMI exact', 'prevprocedure', '#prevprocedures ', 'cysto', 'open',
       ' OR date', 'urine', 'Abx ', 'erectilepre', 'uti', 'LOS (days)', 'spc',
       'tissue', 'Transection ', 'Urethroplasty', 'cath removal', 'cathdays',
       'Failure ', 'Patent', 'fu', 'failure date', 'datetofailureorfollowup',
       'complication', 'clavian II+', 'claviangrade', 'earlycomps',
       'signifcomp', 'minorcomp', 'ER visits', 'foley', 'SP complication',
       'erectilepost', 'Pain ', 'Chordee ', 'PVD ', 'Donor site', 'UTI  Post',
       'UTI  recurring', 'LUTS ', 'Persisting LUTS ', 'Incontinence ',
       'Stricture ', 'Stricture intervention', 'satisfaction', 'Age_calc'],
      dtype='object')

In [11]:
df["cysto"] = df["cysto"].replace(
    ["n/a","none","None", "N/A", "na", "NA", "", None], 0
).fillna(0)
 

/var/folders/cl/h60hyp6s1jg6rr9n0nfgvphw0000gn/T/ipykernel_99954/3180045897.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["cysto"] = df["cysto"].replace(


CLEANING THE COPD COLUMN

NOTES:COPD row 803(no Asthma), n/a1590


In [12]:
#Trying to find the rows that match the notes above
mask = df["COPD "].isin(["n/a", "NaN", "n/a", "N/A","no (Asthma)"])

print(df.loc[mask, "COPD "])

801    no (Asthma)
Name: COPD , dtype: object


In [13]:
mask1 = df["COPD "].astype(str).str.lower().str.contains("n/a|asthma", na=False)
print(df.loc[mask1, "COPD "])


801    no (Asthma)
Name: COPD , dtype: object


In [14]:
print(df.iloc[1595][["COPD "]])  # Only those columns from the 6th row


COPD     1
Name: 1595, dtype: object


In [15]:
print(df["COPD "].unique())

[0 1 'no (Asthma)' 'no ' nan 'on']


In [16]:
mask = ~df["COPD "].isin([0, 1])
print(df.loc[mask, "COPD "])

801     no (Asthma)
881             no 
1588            NaN
1638            NaN
1671            NaN
1860            NaN
1908             on
1916            NaN
Name: COPD , dtype: object


In [17]:
copd_map = {
    "no (asthma)": 0,
    "no": 0,
    "no ": 0,
    "on": 0
}

# Step 1: Lowercase, strip, and replace messy values with 0
df["COPD "] = (
    df["COPD "]
      .astype(str)
      .str.strip()
      .str.lower()
      .replace(copd_map)
)

# Step 2: Convert real np.nan to 0
df["COPD "] = df["COPD "].replace("nan", 0)
df["COPD "] = df["COPD "].replace(np.nan, 0)

# Step 3: Convert to integer (if everything is now numeric)
df["COPD "] = df["COPD "].astype(int)

In [18]:
print(df["COPD "].unique()) #It worked thank God 

[0 1]


CLEANING DIABETES COLUMN

NOTES: Diabetes (n/a 1590)


In [19]:
print(df["Diabetes "].unique())

[1 0 'yes (no meds)' 'no ' nan]


In [20]:

diabetes_map = {
    "yes (no meds)": 1,
    "no": 0,
    "nan ": 0

}
# Step 1: Lowercase, strip, and replace messy values with 0
df["Diabetes "] = (
    df["Diabetes "]
      .astype(str)
      .str.strip()
      .str.lower()
      .replace(diabetes_map)
)

# Step 2: Convert real np.nan to 0
df["Diabetes "] = df["Diabetes "].replace("nan", 0)
df["Diabetes "] = df["Diabetes "].replace(np.nan, 0)

# Step 3: Convert to integer (if everything is now numeric)
df["Diabetes "] = df["Diabetes "].astype(int)


In [21]:
print(df["Diabetes "].unique()) # yesss finallyyyyy

[1 0]


CLEANING TRANSECTIONS COLUMN

Notes: none that I know of

In [22]:
print(df["Transection "].unique())

[0 1 'no']


In [23]:
df["Transection "] = df["Transection "].replace(
    ["no","none","None", "N/A", "na", "NA", "", None], 0
).fillna(0)

/var/folders/cl/h60hyp6s1jg6rr9n0nfgvphw0000gn/T/ipykernel_99954/3125013904.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Transection "] = df["Transection "].replace(


In [24]:
print(df["Transection "].unique()) #It works!!!

[0 1]


CLEANING URETHROPLASTY COLUMN

In [25]:
print(df["Urethroplasty"].unique()) #It works!!!

[ 2.  4.  1.  5.  3.  6. nan]


CLEANING UTI POST COLUMN

NOTES:UTI Post (yes-1, no -0), empty 0?

In [27]:
print(df["UTI  Post"].unique())


['no' 'yes' nan 'No' 'Yes' 'Yes (prostatitis)' 'Yes x 3' 'Yes (E. coli)'
 'Yes (E. coli/ESBL)' 'no fu' 'no FU' 'NO' 'Yes?' '-' 'no follow-up'
 'NO FU' 'not reported' 'lost to f/u' 0 1]


In [30]:
df["UTI  Post"] = df["UTI  Post"].astype(str).str.strip().str.lower()

mask_fu = (
    df["UTI  Post"].str.contains("fu")
    | df["UTI  Post"].str.contains("follow-up")
    | df["UTI  Post"].str.contains("lost")
    | df["UTI  Post"].str.contains("-")
    | df["UTI  Post"].str.contains("not reported")
)
df = df[~mask_fu]

In [31]:
print(df["UTI  Post"].unique())


['no' 'yes' 'nan' 'yes (prostatitis)' 'yes x 3' 'yes (e. coli)'
 'yes (e. coli/esbl)' 'yes?' '0' '1']


In [34]:
pmask = ~df["UTI  Post"].isin(["yes", "no","nan"])
print(df.loc[pmask, "UTI  Post"])

228      yes (prostatitis)
275                yes x 3
388          yes (e. coli)
396     yes (e. coli/esbl)
489                   yes?
               ...        
2045                     0
2046                     0
2047                     0
2051                     0
2058                     0
Name: UTI  Post, Length: 250, dtype: object


In [ ]:
#  map values
def map_uti(val):
    if "no" in val:
        return 0
    if "nan" in val:
        return 0
    if "yes" in val:
        return 1
    if val in ["1", 1]:
        return 1
    if val in ["0", 0]:
        return 0
    return 0  # or 0 if you want to treat ambiguous as 0


In [ ]:
df["UTI  Post"] = df["UTI  Post"].apply(map_uti).astype("Int8")


/var/folders/cl/h60hyp6s1jg6rr9n0nfgvphw0000gn/T/ipykernel_99954/567864168.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["UTI  Post"] = df["UTI  Post"].apply(map_uti).astype("Int8")


In [ ]:
pmask = ~df["UTI  Post"].isin([0, 1])
print(df.loc[pmask, "UTI  Post"]) #It workkksss

Series([], Name: UTI  Post, dtype: Int8)


FOR REFERENCE:
Index(['UAH ID', 'DOB (yyyy-mm-dd)', 'Date of Surgery', 'monthsurg',
       'Age (at time of surgery)', 'Age: <50 = 0; >= 50= 1', 'stxlocation',
       'distal', 'penile', 'StxEtiology', 'StxLength', '# Strictures',
       'charlsons', 'Cormorbidity', 'Diabetes ', 'COPD ', 'Smoker ', 'BMI35+',
       'BMI exact', 'prevprocedure', '#prevprocedures ', 'cysto', 'open',
       ' OR date', 'urine', 'Abx ', 'erectilepre', 'uti', 'LOS (days)', 'spc',
       'tissue', 'Transection ', 'Urethroplasty', 'cath removal', 'cathdays',
       'Failure ', 'Patent', 'fu', 'failure date', 'datetofailureorfollowup',
       'complication', 'clavian II+', 'claviangrade', 'earlycomps',
       'signifcomp', 'minorcomp', 'ER visits', 'foley', 'SP complication',
       'erectilepost', 'Pain ', 'Chordee ', 'PVD ', 'Donor site', 'UTI  Post',
       'UTI  recurring', 'LUTS ', 'Persisting LUTS ', 'Incontinence ',
       'Stricture ', 'Stricture intervention', 'satisfaction', 'Age_calc'],
      dtype='object')

In [41]:
print(df["UTI  recurring"].unique())


['no' nan 'yes (1-entercocci)' 'yes' 'No' 'Yes (ESBL)' 'Yes' 0 'no fu'
 'NO' 'NO FU' 'no FU' '-' 'yes (possible UTI, GrHem)'
 'yes (x1 pseudomonas)' 'o' 'yes @4m' 1 'no Fu']


In [42]:
df["UTI  recurring"] = df["UTI  recurring"].astype(str).str.strip().str.lower()

mask_uti = (
    df["UTI  recurring"].str.contains("fu")
    | df["UTI  recurring"].str.contains("follow-up")
    | df["UTI  recurring"].str.contains("lost")
    | df["UTI  recurring"].str.contains("-")
    | df["UTI  recurring"].str.contains("not reported")
)
df = df[~mask_uti]


/var/folders/cl/h60hyp6s1jg6rr9n0nfgvphw0000gn/T/ipykernel_99954/2356063411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["UTI  recurring"] = df["UTI  recurring"].astype(str).str.strip().str.lower()


In [47]:
print(df["UTI  recurring"].unique())


<IntegerArray>
[0, 1]
Length: 2, dtype: Int8


In [48]:
def map_recur(val):
    if "no" in val:
        return 0
    if "nan" in val:
        return 0
    if "yes" in val:
        return 1
    if "o" in val:
        return 0
    if val in ["1", 1]:
        return 1
    if val in ["0", 0]:
        return 0
    return 0  # or 0 if you want to treat ambiguous as 0

In [50]:
print(df["UTI  recurring"].unique())


<IntegerArray>
[0, 1]
Length: 2, dtype: Int8


CLEANING THE LUTS COLUMN

In [52]:
print(df["LUTS "].unique())


[0 nan 1 'not yet 6 mo' 'yes' 'no']


In [60]:
lmask = ~df["LUTS "].isin(["nan"])
print(df.loc[lmask, "LUTS "])

0         0
1         0
2         0
3         0
4         0
       ... 
2061    NaN
2062    NaN
2063    NaN
2064    NaN
2065    NaN
Name: LUTS , Length: 2003, dtype: object


CLEANING THE ABX

In [62]:
print(df["Abx "].unique())

['yes' nan 'Yes' '?' 'No' 'not available' 'No records'
 'yes (not mentioned)' 'no record' 'yes ' ' yes' 'no' 'YES']


In [69]:
df["Abx "] = df["Abx "].replace(
    ["no","No","No records","no record","not available","?", "na", "NA", "", None], 0
).fillna(0)

In [68]:
print(df["Abx "].unique())

['yes' 0 'Yes' 'yes (not mentioned)' 'yes ' ' yes' 'YES']


In [73]:
def map_abx(val):
    val_str = str(val).strip().lower()
    if "no" in val_str:
        return 0
    if "yes" in val_str:
        return 1
    if val_str in ["", "nan", "n/a", "none"]:
        return 0  # or np.nan if you want to mark missing
    if val_str == "1":
        return 1
    if val_str == "0":
        return 0
    # fallback for anything weird
    return 0

In [74]:
df["Abx "] = df["Abx "].apply(map_abx).astype("Int8")


In [75]:
print(df["Abx "].unique())

<IntegerArray>
[1, 0]
Length: 2, dtype: Int8


In [76]:
print(df["Failure "].unique())

[0 1 nan 2 'not done']
